# Run experiments on NYU HPC (OOD Jupyter)

Trains the configs listed in `RUNS` (cell 3), one per GPU, in parallel. Results and
the experiment log are in `BRANCH_NOTES.md`. Use the overlay-conda kernel (torch installed).

Run the cells top to bottom. Training runs in the background, so you can re-run the
**Monitor** cell any time to check progress. With 2 GPUs, pair runs of similar length:
the session may end when one GPU goes idle (unfinished runs can be continued with `--resume`).

## 1. Unpack the upload
Expects `hpc_bundle.zip` in your scratch folder (uploaded via OOD Files).

In [ ]:
import os, subprocess
SCRATCH = os.environ.get('SCRATCH', f"/scratch/{os.environ['USER']}")
os.chdir(SCRATCH)
if not os.path.isdir('pytorch-cifar-resnet'):
    subprocess.run(['unzip', '-q', 'hpc_bundle.zip'], check=True)
print(os.getcwd(), sorted(os.listdir('.')))

## 2. Check the GPU and install missing packages

In [ ]:
import importlib.util, sys
missing = [p for p, mod in [('pyyaml','yaml'), ('pandas','pandas'), ('wandb','wandb'), ('tqdm','tqdm')]
           if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', '-q', *missing], check=True)
    print('Installed:', missing)
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Start the runs (in the background, one GPU each)

W&B logs online (log in once with `wandb login`). Training output goes to `logs/`.

In [ ]:
os.chdir(os.path.join(SCRATCH, 'pytorch-cifar-resnet'))
os.makedirs('logs', exist_ok=True)
RUNS = ['arch_B_autoaug_e300']          # one config name per GPU
procs = {}
for gpu, name in enumerate(RUNS):
    env = {**os.environ, 'WANDB_MODE': 'online', 'CUDA_VISIBLE_DEVICES': str(gpu)}
    log = open(f'logs/{name}.log', 'w')
    procs[name] = subprocess.Popen(
        [sys.executable, 'main.py', '--config', f'configs/{name}.yaml'],
        stdout=log, stderr=subprocess.STDOUT, env=env)
    print(f'started {name} on GPU {gpu} (pid {procs[name].pid})')

## 4. Monitor (re-run this cell whenever you want an update)

In [ ]:
import re, yaml
def status(name):
    text = open(f'logs/{name}.log', errors='ignore').read()
    cfg = yaml.safe_load(open(f'configs/{name}.yaml'))
    total = cfg['training']['epochs']
    val_size = int(50000 * cfg['data']['val_ratio'])
    epochs = re.findall(r'^Epoch: (\d+)', text, re.M)
    val_accs = [float(a) for a in re.findall(rf'Acc: ([\d.]+)% \(\d+/{val_size}\)', text)] if val_size else []
    best = max(val_accs) if val_accs else None
    finished = 'Best validation accuracy' in text or 'Full training complete' in text
    running = procs.get(name) and procs[name].poll() is None
    state = 'running' if running else ('finished' if finished else 'STOPPED (see log)')
    print(f'{name:24s} best val {f"{best:.2f}%" if best else "-":>7s}'
          f'  (last {f"{val_accs[-1]:.2f}%" if val_accs else "-":>7s})'
          f'  epoch {int(epochs[-1]) + 1 if epochs else 0}/{total}  {state}')
for name in RUNS:
    status(name)

If a run crashed, show the end of its log:

In [ ]:
print(open(f'logs/{RUNS[0]}.log', errors='ignore').read()[-3000:])

## 5. Results (after both runs finish)

In [ ]:
for name in RUNS:
    text = open(f'logs/{name}.log', errors='ignore').read()
    params = re.search(r'Model Parameters: ([\d,]+)', text)
    best = re.search(r'Best validation accuracy: ([\d.]+)%', text)
    print(f'{name:22s} params {params.group(1) if params else "?":>10s}   best val acc {best.group(1) + "%" if best else "not finished"}')

## 6. Continue a run that was cut off (restores weights, optimizer and LR schedule from `last.pth`)

In [ ]:
name, gpu = RUNS[0], 0                  # which run, which GPU
os.chdir(os.path.join(SCRATCH, 'pytorch-cifar-resnet'))
env = {**os.environ, 'WANDB_MODE': 'online', 'CUDA_VISIBLE_DEVICES': str(gpu)}
procs[name] = subprocess.Popen(
    [sys.executable, 'main.py', '--config', f'configs/{name}.yaml', '--resume'],
    stdout=open(f'logs/{name}.log', 'a'), stderr=subprocess.STDOUT, env=env)
print('resumed', name, 'on GPU', gpu)

## 7. Optional: end the session when all runs finish (frees the GPUs)

In [ ]:
import time, shutil
for p in procs.values():
    p.wait()
print('all runs finished at', time.ctime())
job = os.environ.get('SLURM_JOB_ID')
if job and shutil.which('scancel'):
    subprocess.run(['scancel', job])